# Multi-Omics Data Integration

This notebook demonstrates how to integrate multiple omics datasets including:
- Early, intermediate, and late fusion methods
- Similarity Network Fusion (SNF)
- Pathway-level integration
- Joint visualization and analysis

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

# MOAS imports
from backend.omics.integration import (
    EarlyFusion,
    IntermediateFusion,
    SimilarityNetworkFusion,
)

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded!")

## 1. Generate Multi-Omics Data

In [ ]:
# Generate sample multi-omics data
np.random.seed(42)

n_samples = 100
n_genes = 500
n_proteins = 300
n_metabolites = 150

# True sample clusters (hidden structure)
true_clusters = np.array([0] * 30 + [1] * 35 + [2] * 35)
np.random.shuffle(true_clusters)

sample_ids = [f"Sample_{i}" for i in range(n_samples)]

# Transcriptomics data
transcriptomics = pd.DataFrame(
    np.random.randn(n_samples, n_genes),
    index=sample_ids,
    columns=[f"Gene_{i}" for i in range(n_genes)]
)
# Add cluster signal
for i, cluster in enumerate(true_clusters):
    transcriptomics.iloc[i, cluster*50:(cluster+1)*50] += 2

# Proteomics data
proteomics = pd.DataFrame(
    np.random.randn(n_samples, n_proteins),
    index=sample_ids,
    columns=[f"Protein_{i}" for i in range(n_proteins)]
)
for i, cluster in enumerate(true_clusters):
    proteomics.iloc[i, cluster*30:(cluster+1)*30] += 1.5

# Metabolomics data
metabolomics = pd.DataFrame(
    np.random.randn(n_samples, n_metabolites),
    index=sample_ids,
    columns=[f"Metabolite_{i}" for i in range(n_metabolites)]
)
for i, cluster in enumerate(true_clusters):
    metabolomics.iloc[i, cluster*15:(cluster+1)*15] += 1.8

omics_datasets = {
    'transcriptomics': transcriptomics,
    'proteomics': proteomics,
    'metabolomics': metabolomics
}

print("Generated multi-omics datasets:")
for name, df in omics_datasets.items():
    print(f"  {name}: {df.shape}")

## 2. Individual Omics Analysis

In [ ]:
# PCA for each omics type
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, df) in zip(axes, omics_datasets.items(), strict=False):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df)

    pca = PCA(n_components=2)
    coords = pca.fit_transform(X_scaled)

    scatter = ax.scatter(
        coords[:, 0], coords[:, 1],
        c=true_clusters,
        cmap='Set1',
        alpha=0.7
    )
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(f'{name.capitalize()} PCA')

plt.tight_layout()
plt.show()

## 3. Early Fusion (Concatenation)

In [ ]:
# Initialize early fusion
early_fusion = EarlyFusion()

# Fit and transform
early_result = early_fusion.fuse(
    list(omics_datasets.values()),
    n_components=50  # Reduce to 50 components
)

print(f"Early fusion result shape: {early_result.fused_data.shape}")
print(f"Explained variance: {early_result.explained_variance:.2%}")

In [ ]:
# Visualize early fusion result
pca = PCA(n_components=2)
early_coords = pca.fit_transform(early_result.fused_data)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PCA plot
scatter = axes[0].scatter(
    early_coords[:, 0], early_coords[:, 1],
    c=true_clusters,
    cmap='Set1',
    alpha=0.7,
    s=50
)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('Early Fusion - PCA')
axes[0].legend(*scatter.legend_elements(), title='True Cluster')

# Clustering evaluation
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
pred_clusters = kmeans.fit_predict(early_result.fused_data)

ari = adjusted_rand_score(true_clusters, pred_clusters)
silhouette = silhouette_score(early_result.fused_data, pred_clusters)

axes[1].bar(['ARI', 'Silhouette'], [ari, silhouette], color=['#3498db', '#e74c3c'])
axes[1].set_ylim(0, 1)
axes[1].set_title(f'Clustering Performance\nARI: {ari:.3f}, Silhouette: {silhouette:.3f}')

plt.tight_layout()
plt.show()

## 4. Similarity Network Fusion (SNF)

In [ ]:
# Initialize SNF
snf = SimilarityNetworkFusion(k=20, n_iterations=20)

# Compute similarity networks for each omics
similarity_networks = []
for df in omics_datasets.values():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df)
    network = snf.compute_similarity_network(X_scaled)
    similarity_networks.append(network)

# Fuse networks
fused_network = snf.fuse(similarity_networks)

print(f"Individual networks: {len(similarity_networks)}")
print(f"Fused network shape: {fused_network.shape}")

In [ ]:
# Spectral clustering on fused network
spectral = SpectralClustering(
    n_clusters=3,
    affinity='precomputed',
    random_state=42
)
snf_clusters = spectral.fit_predict(fused_network)

# Evaluate
snf_ari = adjusted_rand_score(true_clusters, snf_clusters)
snf_silhouette = silhouette_score(fused_network, snf_clusters, metric='precomputed')

print("SNF Clustering Results:")
print(f"  Adjusted Rand Index: {snf_ari:.3f}")
print(f"  Silhouette Score: {snf_silhouette:.3f}")

In [ ]:
# Visualize fused network
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Individual networks and fused network
titles = ['Transcriptomics', 'Proteomics', 'Metabolomics', 'Fused Network']
networks = similarity_networks + [fused_network]

for ax, network, title in zip(axes, networks, titles, strict=False):
    # Sort by true clusters for visualization
    order = np.argsort(true_clusters)
    sorted_network = network[order][:, order]

    im = ax.imshow(sorted_network, cmap='viridis')
    ax.set_title(title)
    ax.set_xlabel('Samples')
    ax.set_ylabel('Samples')
    plt.colorbar(im, ax=ax, shrink=0.6)

plt.tight_layout()
plt.show()

## 5. Intermediate Fusion (Joint Dimensionality Reduction)

In [ ]:
# Intermediate fusion using joint PCA
intermediate_fusion = IntermediateFusion()

int_result = intermediate_fusion.fuse(
    list(omics_datasets.values()),
    n_components=30
)

print(f"Intermediate fusion result shape: {int_result.fused_data.shape}")

# Visualize
pca = PCA(n_components=2)
int_coords = pca.fit_transform(int_result.fused_data)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    int_coords[:, 0], int_coords[:, 1],
    c=true_clusters,
    cmap='Set1',
    alpha=0.7,
    s=50
)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Intermediate Fusion - Joint PCA')
ax.legend(*scatter.legend_elements(), title='True Cluster')
plt.tight_layout()
plt.show()

## 6. Compare Integration Methods

In [ ]:
# Compare all methods
methods = {
    'Transcriptomics Only': transcriptomics.values,
    'Proteomics Only': proteomics.values,
    'Metabolomics Only': metabolomics.values,
    'Early Fusion': early_result.fused_data,
    'Intermediate Fusion': int_result.fused_data,
}

results = []
for name, data in methods.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(data)

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    pred = kmeans.fit_predict(X_scaled)

    ari = adjusted_rand_score(true_clusters, pred)
    sil = silhouette_score(X_scaled, pred)

    results.append({
        'Method': name,
        'ARI': ari,
        'Silhouette': sil
    })

# Add SNF result
results.append({
    'Method': 'SNF',
    'ARI': snf_ari,
    'Silhouette': snf_silhouette
})

comparison_df = pd.DataFrame(results)
comparison_df

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ARI comparison
colors = ['#95a5a6'] * 3 + ['#3498db'] * 3
axes[0].barh(comparison_df['Method'], comparison_df['ARI'], color=colors)
axes[0].set_xlabel('Adjusted Rand Index')
axes[0].set_title('Clustering Performance (ARI)')
axes[0].axvline(0, color='black', linewidth=0.5)

# Silhouette comparison
axes[1].barh(comparison_df['Method'], comparison_df['Silhouette'], color=colors)
axes[1].set_xlabel('Silhouette Score')
axes[1].set_title('Clustering Performance (Silhouette)')
axes[1].axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\nBest method based on ARI:", comparison_df.loc[comparison_df['ARI'].idxmax(), 'Method'])
print("Best method based on Silhouette:", comparison_df.loc[comparison_df['Silhouette'].idxmax(), 'Method'])

## 7. Omics Contribution Analysis

In [ ]:
# Analyze contribution of each omics type
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

contributions = {}
for name, df in omics_datasets.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df)

    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    scores = cross_val_score(rf, X_scaled, true_clusters, cv=5)
    contributions[name] = scores.mean()

# Plot contributions
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#3498db', '#2ecc71', '#9b59b6']
bars = ax.bar(contributions.keys(), contributions.values(), color=colors)
ax.set_ylabel('Cross-validation Accuracy')
ax.set_title('Individual Omics Predictive Power')
ax.set_ylim(0, 1)

for bar, val in zip(bars, contributions.values(), strict=False):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center')

plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
# Save integrated data
# pd.DataFrame(early_result.fused_data, index=sample_ids).to_csv('results/early_fusion.csv')
# pd.DataFrame(int_result.fused_data, index=sample_ids).to_csv('results/intermediate_fusion.csv')
# pd.DataFrame(fused_network, index=sample_ids, columns=sample_ids).to_csv('results/snf_network.csv')
# comparison_df.to_csv('results/integration_comparison.csv', index=False)

print("Multi-omics integration complete!")
print(f"\nBest integration method: {comparison_df.loc[comparison_df['ARI'].idxmax(), 'Method']}")